In [1]:
import os
import numpy as np
from qiskit import QuantumCircuit

# 1. Definición de datos de prueba (16 valores reducidos para timbre y fase)
# Timbre / Amplitud (valores enteros discretos entre 0 y 15 para 4 bits)
data_t = np.array([1, 3, 7, 12, 15, 8, 4, 2, 0, 5, 10, 14, 9, 6, 11, 13])

# Fase (valores flotantes normalizados entre 0 y 1 o enteros para escalado)
data_p = np.array([0, 2, 5, 9, 13, 11, 6, 1, 4, 8, 14, 15, 10, 3, 7, 12])


def draw_circuit(qc, save_path="circuits/circuit.png"):
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig = qc.draw(output="mpl", fold=30)  # fold ajusta el ancho visual
    fig.savefig(save_path, bbox_inches="tight")
    print(f"Circuito guardado en: {save_path}")


# --- Funciones de encoding ---


def qtse(data):
    data_arr = np.asarray(data, dtype=int).ravel()
    n_qubits = 8
    qc = QuantumCircuit(n_qubits)
    a_qubits = [0, 1, 2, 3]
    t_qubits = [4, 5, 6, 7]

    for qubit in t_qubits:
        qc.h(qubit)

    for t_value, a_value in enumerate(data_arr):
        t_bits = np.binary_repr(int(t_value), width=4)
        a_bits = np.binary_repr(int(a_value), width=4)

        for bit_idx, bit in enumerate(t_bits):
            if bit == "0":
                qc.x(t_qubits[bit_idx])

        for a_idx, abit in enumerate(a_bits):
            if abit == "1":
                qc.mcx(t_qubits, a_qubits[a_idx])

        for bit_idx, bit in enumerate(t_bits):
            if bit == "0":
                qc.x(t_qubits[bit_idx])

    return qc


def qtse_p2(data_a, data_p):
    am_arr = np.asarray(data_a, dtype=int).ravel()
    ph_arr = np.asarray(data_p, dtype=int).ravel()

    n_qubits = 8
    qc = QuantumCircuit(n_qubits)
    a_qubits = [0, 1, 2, 3]
    t_qubits = [4, 5, 6, 7]

    for qubit in t_qubits:
        qc.h(qubit)

    for t_value, (a_value, p_value) in enumerate(zip(am_arr, ph_arr)):
        t_bits = np.binary_repr(int(t_value), width=4)
        a_bits = np.binary_repr(int(a_value), width=4)
        p_bits = np.binary_repr(int(p_value), width=4)

        for bit_idx, bit in enumerate(t_bits):
            if bit == "0":
                qc.x(t_qubits[bit_idx])

        for a_idx, abit in enumerate(a_bits):
            if abit == "1":
                qc.mcx(t_qubits, a_qubits[a_idx])

        for p_idx, pbit in enumerate(p_bits):
            if pbit == "1":
                qc.mcx(t_qubits, a_qubits[p_idx])

        for bit_idx, bit in enumerate(t_bits):
            if bit == "0":
                qc.x(t_qubits[bit_idx])

    return qc


def ry_rz_1(data_t, data_p):
    # Para RYRZ adaptamos la fase a rango [0, 1] si viene como entero
    ph_norm = data_p / 15.0 if np.max(data_p) > 1.0 else data_p
    tm_norm = data_t / 15.0 if np.max(data_t) > 1.0 else data_t

    n_qubits = 5
    qc = QuantumCircuit(n_qubits)
    t_qubits = [1, 2, 3, 4]

    for qubit in t_qubits:
        qc.h(qubit)
    qc.h(0)

    for t_value, (timbre, phase) in enumerate(zip(tm_norm, ph_norm)):
        t_bits = np.binary_repr(t_value, width=4)

        for i, bit in enumerate(t_bits):
            if bit == "0":
                qc.x(t_qubits[i])

        theta_t = timbre * np.pi
        theta_p = phase * np.pi

        qc.mcry(theta_t, t_qubits, 0, None)
        qc.mcrz(theta_p, t_qubits, 0)

        for i, bit in enumerate(t_bits):
            if bit == "0":
                qc.x(t_qubits[i])

    qc.h(0)
    return qc


def qtse_p3(data_a, data_p):
    am_arr = np.asarray(data_a, dtype=int).ravel()
    ph_norm = data_p / 15.0 if np.max(data_p) > 1.0 else data_p

    n_qubits = 8
    qc = QuantumCircuit(n_qubits)
    a_qubits = [0, 1, 2, 3]
    t_qubits = [4, 5, 6, 7]

    for qubit in t_qubits:
        qc.h(qubit)

    for qubit in a_qubits:
        qc.h(qubit)

    for t_value, (a_value, p_value) in enumerate(zip(am_arr, ph_norm)):
        t_bits = np.binary_repr(int(t_value), width=4)
        a_bits = np.binary_repr(int(a_value), width=4)

        for bit_idx, bit in enumerate(t_bits):
            if bit == "0":
                qc.x(t_qubits[bit_idx])

        theta_p = p_value * np.pi

        for a_idx, abit in enumerate(a_bits):
            if abit == "1":
                qc.mcx(t_qubits, a_qubits[a_idx])
                qc.mcrz(theta_p, t_qubits, a_qubits[a_idx])

        for bit_idx, bit in enumerate(t_bits):
            if bit == "0":
                qc.x(t_qubits[bit_idx])

    for qubit in a_qubits:
        qc.h(qubit)

    return qc


# --- Generación y guardado de las 4 imágenes ---

if __name__ == "__main__":
    # 1. QTSE (solo timbre/amplitud)
    qc_qtse = qtse(data_t)
    draw_circuit(qc_qtse, "circuits/qtse.png")

    # 2. QTSE_P (QTSE_P2 - Timbre + Fase con MCX)
    qc_qtse_p = qtse_p2(data_t, data_p)
    draw_circuit(qc_qtse_p, "circuits/qtse_p.png")

    # 3. RYRZ
    qc_ryrz = ry_rz_1(data_t, data_p)
    draw_circuit(qc_ryrz, "circuits/ryrz.png")

    # 4. QTSE_RZ (QTSE_P3 - Timbre con MCX + Fase con MCRZ)
    qc_qtse_rz = qtse_p3(data_t, data_p)
    draw_circuit(qc_qtse_rz, "circuits/qtse_rz.png")

Circuito guardado en: circuits/qtse.png
Circuito guardado en: circuits/qtse_p.png
Circuito guardado en: circuits/ryrz.png
Circuito guardado en: circuits/qtse_rz.png
